In [104]:
import random
import json

In [105]:
dataset = json.load(open("locomo10.json", "r"))

In [106]:
def format_session(session, timestamp):
    session_conv = f"TIMESTAMP: {timestamp}\nCONVERSATION:\n"
    for dialogue in session:
        utterance = ""
        if dialogue.get("blip_caption") is not None:
            utterance += f"<image_description>{dialogue['blip_caption']}</image_description>"
        utterance += dialogue['text']
        session_conv += f"\t{dialogue['speaker']}: {utterance}\n"
    return session_conv + '\n'

In [107]:
def format_single_hop(evidence, conversation):
    session_no = int(evidence[0].strip('D').split(':')[0])
    session_id = f"session_{session_no}"
    session = conversation.get(session_id)
    session_date = conversation.get(f"{session_id}_date_time")
    context_conversation = format_session(session, session_date)
    return context_conversation

In [108]:
def format_multihop(conversation):
    session_ids = [item for i in range(len(conversation)) for item in conversation.keys() if item == 'session_%d' % (i+1)]
    full_conv = ""
    for session_id in session_ids:
        session_date = conversation[f"{session_id}_date_time"]
        session = conversation[session_id]
        session_conv = format_session(session, session_date)
        full_conv += session_conv 
    return full_conv

In [109]:
def add_evidence(examples):
    qas = examples['qa']
    conversation = examples['conversation']
    for qa in qas:
        evidence = qa['evidence']
        session_ids = [item.split(":")[0].strip("D") for item in qa['evidence']]
        session_ids = list(set(['session_' + item for item in session_ids if item.isdigit()]))
        sessions = [conversation[session_id] for session_id in session_ids]
        dialogue = []
        for session in sessions:
            for item in session:
                if item.get("dia_id") in set(evidence):
                    text = f"{item['speaker']}: {item['text']}"
                    if text not in dialogue:
                        dialogue.append(text.strip())
                
        qa['dialogue'] = dialogue
    examples['qa'] = qas
    return examples

In [110]:
def remove_session(examples):
    conversation = examples['conversation']
    session_ids = [item for i in range(len(conversation)) for item in conversation.keys() if item == 'session_%d' % (i+1)]
    sessions = [item for item in conversation.keys() if 'date_time' in item and '_'.join(item.split('_')[:2]) not in set(session_ids)]
    conversation = {
        k: v
        for k, v in conversation.items()
        if not k in set(sessions)
    }
    examples['conversation'] = conversation
    return examples

In [111]:
PROMPT = """Answer the question based on the following conversation.

{conversation}

Question: {question}

Answer with exact words from the conversations whenever possible and do not write lengthy and complete sentences.
If the answer cannot be found, do not share false information.
"""

In [ ]:
def format_dataset(examples, category='all'):
    examples = add_evidence(remove_session(examples))
    if category == 'all':
        qas = examples['qa']
        categories = []
    else:
        qas = [example for example in examples['qa'] if example['category'] == category]
        categories = None

    conversation = examples['conversation']

    questions = []
    prompts = []
    answers = []
    dialogue = []
    for qa in qas:
        if qa['category'] == 4:
            context_conv = format_single_hop(qa['evidence'], conversation)
        else:
            context_conv = format_multihop(conversation)
        query = qa['question']
        dialogue.append(qa['dialogue'])
        questions.append(query)

        if qa['category'] == 2:
            query += " Use the timestamps as a reference to approximate the answer when needed."
        if qa['category'] == 5:
            choices = "\nChoose the correct answer:\n(a) {}\n(b){}"
            if random.random() < 0.5:
                query += choices.format(qa['adversarial_answer'], "No information available")
                answer = {'a': qa['adversarial_answer'], 'b': "No information available"}
            else:
                query += choices.format("No information available", qa['adversarial_answer'])
                answer = {'a': "No information available", 'b': qa['adversarial_answer']}

        else:
            answer = str(qa['answer'])

        prompt = PROMPT.format(conversation=context_conv, question=query)
        prompt = [
            {"role": "user", "content": prompt}
        ]
        
        prompts.append(prompt)
        answers.append(answer)
        if categories is not None:
            categories.append(qa['category'])

    dataset = {
        'question': questions,
        "prompt": prompts,
        "answer": answers,
    }

    if not category == 5:
        dataset['dialogue'] = dialogue

    if categories is not None:
        dataset['category'] = categories
    return dataset

In [113]:
format_dataset(dataset[0], 1)['answer']

['Adoption agencies',
 'Transgender woman',
 'Single',
 'Sweden',
 'counseling or mental health for Transgender people',
 'pottery, camping, painting, swimming',
 'beach, mountains, forest',
 'dinosaurs, nature',
 '"Nothing is Impossible", "Charlotte\'s Web"',
 'Running, pottery',
 'Pride parade, school speech, support group',
 'Mentoring program, school speech',
 'sunset',
 'Pottery, painting, camping, museum, swimming, hiking',
 'Joining activist group, going to pride parades, participating in an art show, mentoring program',
 2,
 'abstract art',
 'Her mentors, family, and friends',
 'bowls, cup',
 'Horse, sunset, sunrise',
 'Oliver, Luna, Bailey',
 'Sunsets',
 'Rainbow flag, transgender symbol',
 'clarinet and violin',
 'Summer Sounds, Matt Patterson',
 'Changes to her body, losing unsupportive friends',
 'Roast marshmallows, tell stories',
 'Poetry reading, conference',
 '"Becoming Nicole"',
 3,
 '19 October 2023',
 'Figurines, shoes']